# Actividad 1: Pipeline Concurrente

**Objetivo:** Implementar un pipeline que reciba solicitudes concurrentes para procesar imágenes y realice inferencias usando un modelo de aprendizaje automático (MobileNetV2). Se simula la concurrencia con `asyncio`.

## 1. Importar librerías y cargar el modelo

Cargamos MobileNetV2 preentrenada en ImageNet. Esta red espera imágenes de tamaño 224x224 píxeles con 3 canales (RGB).

In [1]:
import asyncio
import numpy as np
import tensorflow as tf

# Cargar el modelo de IA preentrenado (solo una vez, para reutilizarlo)
print("Cargando modelo MobileNetV2...")
model = tf.keras.applications.MobileNetV2(weights="imagenet")
print("Modelo cargado correctamente.")

2026-04-24 12:05:52.175303: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Cargando modelo MobileNetV2...


2026-04-24 12:05:59.949593: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Modelo cargado correctamente.


## 2. Procesamiento de una sola imagen

La función `process_request` simula el trabajo (espera asíncrona de 1 segundo) y luego realiza la inferencia con el modelo.

In [3]:
async def process_request(image_data):
    """
    Procesa una imagen de forma asíncrona.
    
    Parámetros:
    - image_data: numpy array con forma (1, 224, 224, 3)
    
    Retorna:
    - Predicción del modelo (vector de probabilidades)
    """
    print("  -> Iniciando procesamiento de una solicitud...")
    # Simular tiempo de cómputo (operación I/O o cálculo pesado)
    await asyncio.sleep(1)  # En un caso real, aquí se haría una llamada a un servicio externo
    # Inferencia (operación bloqueante, pero se ejecuta en el hilo principal)
    prediction = model.predict(image_data, verbose=0)
    print("  -> Solicitud completada.")
    return prediction

## 3. Pipeline concurrente

Creamos una tarea asíncrona por cada imagen y ejecutamos todas en paralelo con `asyncio.gather`.

In [4]:
async def pipeline_concurrente(requests):
    """
    Ejecuta múltiples solicitudes de forma concurrente.
    
    Parámetros:
    - requests: lista de arrays de imágenes
    
    Retorna:
    - Lista de predicciones en el mismo orden que las solicitudes.
    """
    # Crear una tarea asíncrona por cada solicitud
    tasks = [asyncio.create_task(process_request(img)) for img in requests]
    # Esperar a que todas terminen y recolectar resultados
    results = await asyncio.gather(*tasks)
    return results

## 4. Generar datos de prueba

Simulamos `num_images` imágenes aleatorias con la forma esperada por MobileNetV2: (1, 224, 224, 3).  
Normalmente se deberían preprocesar (escalar píxeles, etc.), pero para este ejemplo solo usamos valores aleatorios.

In [5]:
def simulate_data(num_images):
    """
    Genera una lista de imágenes sintéticas.
    
    Parámetros:
    - num_images: número de imágenes a generar
    
    Retorna:
    - Lista de arrays de forma (1, 224, 224, 3)
    """
    # Nota: la forma correcta es (224,224,3) no (224,24,3) como aparecía en el PDF
    return [np.random.rand(1, 224, 224, 3) for _ in range(num_images)]

## 5. Ejecución del pipeline

En Jupyter no se puede usar `asyncio.run()` directamente porque el event loop ya está corriendo.  
Aplicamos `nest_asyncio` para permitir bucles anidados y luego ejecutamos `await main()`.

In [6]:
async def main():
    # Simular 5 solicitudes de imágenes
    data = simulate_data(5)
    print(f"Se simularon {len(data)} imágenes.")
    
    # Medir tiempo de ejecución (opcional)
    import time
    start = time.time()
    
    results = await pipeline_concurrente(data)
    
    end = time.time()
    print(f"\nPipeline completado en {end - start:.2f} segundos.")
    print("Resultados procesados (primeros 5 valores de cada predicción):")
    for i, pred in enumerate(results):
        # Mostrar solo las primeras 5 clases con mayor probabilidad
        top5 = np.argsort(pred[0])[-5:][::-1]
        print(f"Imagen {i+1}: predicciones (índices top-5): {top5}")

In [7]:
# Parche necesario para ejecutar asyncio en Jupyter
import nest_asyncio
nest_asyncio.apply()

# Ejecutar la función principal
await main()

Se simularon 5 imágenes.
  -> Iniciando procesamiento de una solicitud...
  -> Iniciando procesamiento de una solicitud...
  -> Iniciando procesamiento de una solicitud...
  -> Iniciando procesamiento de una solicitud...
  -> Iniciando procesamiento de una solicitud...
  -> Solicitud completada.
  -> Solicitud completada.
  -> Solicitud completada.
  -> Solicitud completada.
  -> Solicitud completada.

Pipeline completado en 2.91 segundos.
Resultados procesados (primeros 5 valores de cada predicción):
Imagen 1: predicciones (índices top-5): [885 911 619 844 700]
Imagen 2: predicciones (índices top-5): [885 911 619 700 844]
Imagen 3: predicciones (índices top-5): [885 911 619 700 539]
Imagen 4: predicciones (índices top-5): [885 911 619 844 892]
Imagen 5: predicciones (índices top-5): [885 911 619 700 844]


## Observaciones sobre la Actividad 1

- **Concurrencia real vs paralelismo**: `asyncio` permite manejar muchas tareas bloqueantes (ej. esperas de red) sin hilos, pero `model.predict()` es una operación CPU‑intensiva que bloqueará el event loop. En producción se usaría un `ThreadPoolExecutor` o se dejaría la inferencia en otro proceso.
- **Escalado**: Para 100 solicitudes concurrentes (primer reto), el tiempo total será cercano al de la inferencia más lenta, ya que se ejecutan en paralelo. Sin embargo, el modelo no está preparado para procesar lotes grandes simultáneamente; se debería usar `predict` con batch.
- **Mejora propuesta**: Modificar `process_request` para aceptar lotes de imágenes y usar `model.predict(batch_images)`.

Ejecuta el código anterior y verás que las 5 imágenes se procesan en ~1 segundo (más el tiempo de inferencia).